Import libraries

In [1]:
import json
import numpy as np

from pathlib import Path
from qdrant_client import QdrantClient, models

print("Libraries imported successfully!")

Libraries imported successfully!


Load chunks and embeddings

In [2]:
PROJECT_ROOT = Path.cwd().parent

CHUNK_FILE = (
    PROJECT_ROOT / "knowledge_base" / "processed"
    / "chunked_documents.jsonl"
)

EMBEDDING_FILE = (
    PROJECT_ROOT / "knowledge_base" / "processed"
    / "chunk_embeddings.npy"
)

chunked_documents = []

with open(CHUNK_FILE, "r", encoding="utf-8") as file:
    for line in file:
        chunked_documents.append(json.loads(line))

embeddings = np.load(EMBEDDING_FILE)

print("Total chunks:", len(chunked_documents))
print("Embedding shape:", embeddings.shape)

assert len(chunked_documents) == len(embeddings)

Total chunks: 1017
Embedding shape: (1017, 384)


Initialize Qdrant

In [3]:
QDRANT_PATH = PROJECT_ROOT / "qdrant_storage"

client = QdrantClient(path=str(QDRANT_PATH))

COLLECTION_NAME = "factorymind_knowledge"

print("Qdrant initialized successfully!")

Qdrant initialized successfully!


Create a Qdrant collection

In [4]:
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=384,
        distance=models.Distance.COSINE
    )
)

print("Qdrant collection created!")

Qdrant collection created!


Store embeddings and document metadata

points = []

for index, chunk in enumerate(chunked_documents):

    payload = {
        "chunk_id": chunk["chunk_id"],
        "source": chunk["source"],
        "page": chunk["page"],
        "chunk_index": chunk["chunk_index"],
        "text": chunk["text"]
    }

    points.append(
        models.PointStruct(
            id=index,
            vector=embeddings[index].tolist(),
            payload=payload
        )
    )

client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
    wait=True
)

print("All vectors stored successfully!")

Verify stored vectors

collection_info = client.get_collection(COLLECTION_NAME)

print("Collection:", COLLECTION_NAME)
print("Stored vectors:", collection_info.points_count)
print("Vector dimensions:", 384)

Test semantic search

In [5]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

query = "What are the best practices for industrial equipment maintenance?"

query_vector = embedding_model.encode(
    query,
    normalize_embeddings=True
).tolist()

search_results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=3,
    with_payload=True
).points

for result in search_results:

    print("\nSimilarity score:", result.score)
    print("Source:", result.payload["source"])
    print("Page:", result.payload["page"])
    print("Content:")
    print(result.payload["text"][:500])
    print("-" * 60)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
client.close()
print("Qdrant client closed successfully!")

Qdrant client closed successfully!
